# Ejemplo 22: Representaciones del Grupo de Lorentz

Cuaderno de apoyo para `Tutorial/02_relatividad_y_campos/03_representaciones_de_lorentz_y_espinores.md`

**Objetivo:** construir explicitamente boosts y rotaciones de Lorentz, verificar sus propiedades algebraicas y entender la diferencia entre las representaciones escalar, vectorial y espinorial.

In [ ]:
import numpy as np
import sympy as sp

## Ejemplo 1: Boost de Lorentz en representacion vectorial

Un boost en la direccion $x$ con velocidad $\beta = v/c$ y factor $\gamma = 1/\sqrt{1-\beta^2}$ transforma un 4-vector $(t, x, y, z)$ como:

$$\Lambda^\mu{}_\nu = \begin{pmatrix}\gamma & -\gamma\beta & 0 & 0 \\ -\gamma\beta & \gamma & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1\end{pmatrix}.$$

In [ ]:
def boost_x(beta):
    """Matriz de boost de Lorentz en la direccion x."""
    gamma = 1.0 / np.sqrt(1 - beta**2)
    return np.array([
        [gamma,        -gamma*beta, 0, 0],
        [-gamma*beta,  gamma,       0, 0],
        [0,            0,           1, 0],
        [0,            0,           0, 1]
    ])

# Verificacion: preserva el intervalo (t^2 - x^2 - y^2 - z^2)
g = np.diag([1, -1, -1, -1])  # metrica Minkowski

beta = 0.6
L = boost_x(beta)
LgL = L.T @ g @ L

print(f"Boost con beta = {beta}, gamma = {1/np.sqrt(1-beta**2):.4f}")
print("\nLambda^T g Lambda (debe ser igual a g):")
print(np.round(LgL, 6))
print("\nVerificacion: Lambda preserva la metrica de Minkowski.")

## Ejemplo 2: Composicion de boosts y no conmutatividad

Dos boosts en distintas direcciones NO conmutan. Su composicion produce ademas una rotacion (precesion de Thomas).

In [ ]:
def boost_y(beta):
    """Boost en la direccion y."""
    gamma = 1.0 / np.sqrt(1 - beta**2)
    return np.array([
        [gamma,  0, -gamma*beta, 0],
        [0,      1,  0,          0],
        [-gamma*beta, 0, gamma,  0],
        [0,      0,  0,          1]
    ])

beta = 0.3
Lx = boost_x(beta)
Ly = boost_y(beta)

LxLy = Lx @ Ly
LyLx = Ly @ Lx

conmutador = LxLy - LyLx
print("Conmutador [Lambda_x, Lambda_y] = Lambda_x Lambda_y - Lambda_y Lambda_x:")
print(np.round(conmutador, 6))
print("\nEl conmutador NO es cero: los boosts en distintas direcciones no conmutan.")
print("La composicion genera una rotacion (precesion de Thomas).")

## Ejemplo 3: Generadores del algebra de Lorentz

El algebra de Lorentz $\mathfrak{so}(1,3)$ tiene 6 generadores: 3 rotaciones $J_i$ y 3 boosts $K_i$. Satisfacen:

$$[J_i, J_j] = i\varepsilon_{ijk}J_k, \quad [J_i, K_j] = i\varepsilon_{ijk}K_k, \quad [K_i, K_j] = -i\varepsilon_{ijk}J_k.$$

In [ ]:
# Generadores de boost K_x, K_y, K_z (representacion vectorial 4x4)
Kx = np.array([[0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], dtype=float)
Ky = np.array([[0, 0, 1, 0], [0, 0, 0, 0], [1, 0, 0, 0], [0, 0, 0, 0]], dtype=float)
Kz = np.array([[0, 0, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 0, 0]], dtype=float)

# Generadores de rotacion J_x, J_y, J_z
Jx = np.array([[0,0,0,0],[0,0,0,0],[0,0,0,-1],[0,0,1,0]], dtype=float)
Jy = np.array([[0,0,0,0],[0,0,0,1],[0,0,0,0],[0,-1,0,0]], dtype=float)
Jz = np.array([[0,0,0,0],[0,0,-1,0],[0,1,0,0],[0,0,0,0]], dtype=float)

def conm(A, B):
    return A @ B - B @ A

# Verificar [Kx, Ky] = -i Jz
print("[Kx, Ky] (debe ser -Jz en convencion con factor i absorbido):")
print(np.round(conm(Kx, Ky), 6))
print("-Jz:")
print(np.round(-Jz, 6))
print("\n[Jx, Jy] (debe ser Jz):")
print(np.round(conm(Jx, Jy), 6))
print("Jz:")
print(np.round(Jz, 6))

## Ejemplo 4: Representacion espinorial vs vectorial

La representacion espinorial de $SL(2,\mathbb{C})$ es la representacion fundamental 2-dimensional. Un boost en x actua sobre espinores de Weyl como $e^{\eta\sigma_x/2}$ donde $\eta = \text{arctanh}(\beta)$ es la rapidez.

In [ ]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def boost_espinor_x(beta):
    """Boost en x sobre espinor de Weyl."""
    eta = np.arctanh(beta)  # rapidez
    # exp(eta * sigma_x / 2) = cosh(eta/2) I + sinh(eta/2) sigma_x
    return np.cosh(eta/2) * I2 + np.sinh(eta/2) * sigma_x

beta = 0.6
M = boost_espinor_x(beta)
print(f"Boost espinorial con beta = {beta}:")
print(np.round(M, 4))
print()

# Un espinor transforma con M, un vector transforma con Lambda (4x4)
# La clave: rotar 360 grados un espinor da -espinor (doble cubrimiento)
theta = 2 * np.pi  # rotacion de 360 grados
R360_espinor = np.cos(theta/2) * I2 + 1j * np.sin(theta/2) * sigma_z
print("Rotacion 360 grados sobre espinor (debe dar -I):")
print(np.round(R360_espinor.real, 6))
print("=> Espinores adquieren un signo menos bajo rotacion de 2pi: son objetos de espin 1/2.")

## Resumen

| Representacion | Dimension | Objeto fisico | Rotacion 2pi |
|---|---|---|---|
| Escalar $(0,0)$ | 1 | Campo de Higgs | $+1$ |
| Espinor izq $(1/2,0)$ | 2 | Weyl izquierdo $\psi_L$ | $-1$ |
| Espinor der $(0,1/2)$ | 2 | Weyl derecho $\psi_R$ | $-1$ |
| Vectorial $(1/2,1/2)$ | 4 | Campo $A_\mu$ | $+1$ |
| Dirac $(1/2,0)\oplus(0,1/2)$ | 4 | Fermiion de Dirac | $-1$ |